# pandas: `DataFrame` reshaping

Previously, I combined a number of Mega Millions lottery `DataFrame` instances into a single `DataFrame` covering the period 2002-2024. I then joined the `DataFrame` to a second `DataFrame` in order to add an additional five columns representing each of the lottery's "Pick 5" winning numbers. In this lesson I'll demonstrate how to reshape the lottery data to help determine how often individual numbers appear among each lottery draw's winning numbers.

In [1]:
import altair as alt
import numpy as np
import pandas as pd

## Retrieve and prep the data

First, let's retrieve the lottery data and then call `DataFrame.head()` to refamilarize ourselves with layout of the frame.

In [2]:
data = pd.read_csv("./data/mega_millions_pick5_cols-2002_24.csv")
data.head(3)

,draw_date,winning_numbers,mega_ball,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5
0,05/17/2024,08 17 40 60 70,3,2.0,8,17,40,60,70
1,05/14/2024,13 19 43 62 64,6,3.0,13,19,43,62,64
2,05/10/2024,13 22 26 32 65,18,4.0,13,22,26,32,65


Next, I'll convert each "draw_date" string to a `datetime64[ns]` object to simplify working with the lottery draw dates. We can confirm that the type conversion operation was successful by checking the `data` object's `dtypes` property.

In [3]:
data["draw_date"] = pd.to_datetime(data.loc[:, "draw_date"], format="%m/%d/%Y")
data.dtypes

draw_date          datetime64[ns]
winning_numbers            object
mega_ball                   int64
multiplier                float64
pick5_1                     int64
pick5_2                     int64
pick5_3                     int64
pick5_4                     int64
pick5_5                     int64
dtype: object

## Reshape the data

Let's determine the frequency of individual Pick 5 winning numbers. For example, how often has the number `31` figured among the winning numbers? My working hypothesis is that each winning number is drawn randomly and the selection frequency is distributed evenly across the number range.

### Wide format vs long format

The lottery data stored in `data` is derived from a _wide format_ CSV file (think horizontal) in which each row represents a single draw date in which the associated Mega ball, Pick 5, and multiplier values are each stored separately in columns.

An alternative structure is the _long format_ (think vertical) in which each value associated with a draw date is stored in its own row. The number of **rows** required to represent a winning draw increases from `1` to `8`, while the number of **columns** required is reduced to `3`:

| Index | draw_date | variable | value |
| :---- | :-------- | :------- | :---- |
| 0 | 2024-05-17 | mega_ball | 3 |
| 1	| 2024-05-17 | multiplier | 2.0 |
| 2	| 2024-05-17 | pick5_1 | 8 |
| 3	| 2024-05-17 | pick5_2 | 17 |
| 4	| 2024-05-17 | pick5_3 | 40 |
| 5	| 2024-05-17 | pick5_4 | 60 |
| 6	| 2024-05-17 | pick5_5| 70 |

Switching to a long format should simplify obtaining a frequency count of each of the winning numbers that are currently stored across five "pick5_*" columns in `data`.

### Melting

I can reshape the data from a wide to a long format by calling either the [`pd.melt()`](https://pandas.pydata.org/docs/reference/api/pandas.melt.html) function or the [`DataFrame.melt()`](https://pandas.pydata.org/docs/reference/api/pandas.melt.html) method. The `melt()` callables are designed to unpivot a `DataFrame` from wide to long format.

`pandas.melt(frame, id_vars=None, value_vars=None, var_name=None, value_name='value', col_level=None, ignore_index=True)`

`DataFrame.melt(id_vars=None, value_vars=None, var_name=None, value_name='value', col_level=None, ignore_index=True)`

In the example below I call the `DataFrame.melt()` method to coerce the lottery data into a long format, three column `DataFrame`, by passing the following keyword arguments to the method:

* `id_vars`: column(s) that serve as identifier variable(s) (e.g., "draw_date")
* `value_vars`: columns to unpivot (e.g., all columns except "draw_date")
* `var_name`: name of the "variable" column
* `value_name`: name of the "value" column (current column labels cannot be used)

The new `DataFrame` is then sorted by "draw_date" (ascending order) and "variable" (descending order) columns.

In [4]:
columns = data.columns.tolist()  # column names
data_long = data.melt(
    id_vars=["draw_date"], value_vars=columns[1:], var_name="variable", value_name="value"
).sort_values(by=["draw_date", "variable"], ascending=[False, True], ignore_index=True)

data_long.head(16)

,draw_date,variable,value
0,2024-05-17,mega_ball,3
1,2024-05-17,multiplier,2.0
2,2024-05-17,pick5_1,8
3,2024-05-17,pick5_2,17
4,2024-05-17,pick5_3,40
5,2024-05-17,pick5_4,60
6,2024-05-17,pick5_5,70
7,2024-05-17,winning_numbers,08 17 40 60 70
8,2024-05-14,mega_ball,6
9,2024-05-14,multiplier,3.0


## Value counts

The long format facilitates computing the frequency of each distinct "pick5_*" row in the `DataFrame`. All I need to do is apply a Boolean mask when counting each winning number to exclude non Pick 5 rows.

In [5]:
mask = data_long.loc[:, "variable"].str.startswith("pick5_")
counts = data_long[mask]["value"].value_counts(sort=True).reset_index()  # returns DataFrame
counts

,value,count
0,31,219
1,10,217
2,20,213
3,17,212
4,14,211
...,...,...
70,73,31
71,74,30
72,75,25
73,71,22


Alternatively, I could forgo the melt step and call the [`DataFrame.stack()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.stack.html) method to start the process of obtaining the counts. The `stack()` method takes values stored horizontally and stacks them vertically. The method is optimized to work with `DataFrame` objects that feature a multi-level, hierarchial `MultiIndex` but it also works with a single level column axis.

`DataFrame.stack(level=-1, dropna=_NoDefault.no_default, sort=_NoDefault.no_default, future_stack=False)`

Below I pass a "Pick 5" slice of `data` (wide format) to the `stack()` method. The return value is a `Series`. I then method chain my way to success by counting the values, resetting the index, and then renaming the column label "index" to "value". I do all this so that I can confirm that both of my "count" `Series` are equal.

In [6]:
counts_stack = (
    data.loc[:, "pick5_1":"pick5_5"]
    .stack()
    .value_counts(sort=True)
    .reset_index()
    .rename({"index": "value"}, axis="columns")
)
counts_stack

,value,count
0,31,219
1,10,217
2,20,213
3,17,212
4,14,211
...,...,...
70,73,31
71,74,30
72,75,25
73,71,22


In [7]:
pd.testing.assert_frame_equal(counts, counts_stack)

### Plot it

The output above reveals that the number `31` has appeared `219` times among the Pick 5 winning numbers while the number `72` has only figured in `20` Pick 5 winning numbers. Let's visualize the frequency counts with a bar chart for a more enlightening view.

💡 I'm using the [Vega-Altair](https://altair-viz.github.io/) library to visualize the data.

In [8]:
base = alt.Chart(counts)  # Altair Chart requires DataFrame
bar = base.mark_bar(size=3).encode(
    x=alt.X(
        "value:Q",
        axis=alt.Axis(labelAngle=0, tickCount=15, tickMinStep=5),
        bin=False,
        scale=alt.Scale(domain=(1, 75)),
        sort="-y",
        title="Pick 5 numbers",
    ),
    y=alt.Y("count:Q", scale=alt.Scale(domain=(0, 220)), title="Count"),
    color=alt.Color("count:Q", scale=alt.Scale(scheme="blues")),
)
rule = base.mark_rule().encode(
    alt.Y("mean(count):Q"),
    color=alt.value("red"),
    strokeWidth=alt.value(1),
)
(bar + rule).properties(
    title=alt.Title("Pick 5 winning number frequencies, 2002-2024", align="center")
)

alt.LayerChart(...)

The bar chart reveals a decided drop in winning number counts for numbers between `57` and `75`. It's a surprising finding.

Perhaps this is evidence that the Mega Millions winning number drawing machine is defective or the game is rigged in some way. I recommend that we withhold judgement. As a data scientist be wary of opting for such a conclusion based on a single (limited) set of observations.

## A bit of history (know your data)

The range of Mega Ball and Pick 5 numbers have fluctuated over the years as the multi-state [Mega Millions](https://en.wikipedia.org/wiki/Mega_Millions) lottery rules have been tweaked. The current iteration of the game has been in effect since 28 October 2017. Pick 5 numbers now range from `1` to `70` while Mega Ball number range from `1` to `25`. Earlier iterations of the game featured shifting Mega Ball number ranges as well as Pick 5 numbers that ranged between `1` to `50`, `1` to `56`, and `1` to `75`.

Given the shifting format and undoubted impact on player number selections, it's unwise to work with the data set in its entirety&mdash;at least with respect to my question&mdash;without factoring in the history of the game. Given this, let's focus our exploration on the current Pick 5/70, Pick 1/25 (Mega Ball) format. This will entail dropping all rows featuring a "draw_date" that's less than 28 October 2017.

To accomplish the task, I'll leverage the [`DataFrame.drop()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html) method which removes rows or columns along a specified axis. The method permits in-place drop operations on an existing `DataFrame` via the `inplace=< bool >` parameter.

In [9]:
mask = data_long["draw_date"] <= pd.Timestamp(2017, 10, 28)
data_long.drop(index=data_long[mask].index, inplace=True)
data_long.shape

(5472, 3)

Next, I'll (re)compute the value counts and plot the results. The varying frequency counts may still raise questions but we will leave that issue for another lesson.

In [10]:
mask = data_long.loc[:, "variable"].str.startswith("pick5_")
counts = data_long[mask]["value"].value_counts(sort=True).reset_index()  # returns DataFrame

In [11]:
base = alt.Chart(counts)  # Altair Chart requires DataFrame
bar = base.mark_bar(size=3).encode(
    x=alt.X(
        "value:Q",
        axis=alt.Axis(labelAngle=0, tickCount=15, tickMinStep=5),
        bin=False,
        scale=alt.Scale(domain=(1, 70)),
        sort="-y",
        title="Pick 5 numbers",
    ),
    y=alt.Y("count:Q", scale=alt.Scale(domain=(0, 70)), title="Count"),
    color=alt.Color("count:Q", scale=alt.Scale(scheme="blues")),
)
rule = base.mark_rule().encode(
    alt.Y("mean(count):Q"),
    color=alt.value("red"),
    strokeWidth=alt.value(1),
)
(bar + rule).properties(
    title=alt.Title("Pick 5 winning number frequencies, 2017-2024", align="center")
)

alt.LayerChart(...)

## Pivot the data

I commenced this lesson with a "wide format" `DataFrame` named `data`. I leveraged the frame's `melt()` method to create a "long format" version of the `DataFrame` so that I could explore the frequency of individual Pick 5 winning numbers. After observing that certain low count winning numbers no longer feature in the game I removed row data to concentrate on the latest iteration of the game.

I'd like to persist `data_long` to a file but I'd prefer to revert back to a wide format representation of the lottery data covering the years 2017-2024. I can reshape the data by calling the [`DataFrame.pivot()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html) method. The `pivot()` method reshapes a `DataFrame` based on its column values.

`DataFrame.pivot(*, columns, index=_NoDefault.no_default, values=_NoDefault.no_default)`

You select the columns to use for the new frame's columns, values, and index. If the latter is not specified the existing index will be utilized.

Let's review the columns available in `data_long`:

In [12]:
data_long.head(8)

,draw_date,variable,value
0,2024-05-17,mega_ball,3
1,2024-05-17,multiplier,2.0
2,2024-05-17,pick5_1,8
3,2024-05-17,pick5_2,17
4,2024-05-17,pick5_3,40
5,2024-05-17,pick5_4,60
6,2024-05-17,pick5_5,70
7,2024-05-17,winning_numbers,08 17 40 60 70


From `data_long` I'll specify "draw_date" as the index, derive the columns from the "variable" column values, and populate the new columns with values sourced from the "value" column. I'll assign the new `DataFrame` to the variable named `data_wide`.

In [13]:
data_wide = data_long.pivot(index="draw_date", columns="variable", values="value")
data_wide

variable,mega_ball,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5,winning_numbers
draw_date,,,,,,,,
2017-10-31,12,4.0,6,28,31,52,53,06 28 31 52 53
2017-11-03,3,2.0,10,22,42,61,69,10 22 42 61 69
2017-11-07,11,4.0,1,54,60,68,69,01 54 60 68 69
2017-11-10,24,2.0,6,23,38,42,58,06 23 38 42 58
2017-11-14,19,3.0,1,14,21,22,28,01 14 21 22 28
...,...,...,...,...,...,...,...,...
2024-05-03,11,2.0,6,13,15,53,56,06 13 15 53 56
2024-05-07,15,3.0,26,28,36,63,66,26 28 36 63 66
2024-05-10,18,4.0,13,22,26,32,65,13 22 26 32 65


Looks good, but let's check the dtypes. Bummer, all objects.

In [14]:
data_wide.dtypes

variable
mega_ball          object
multiplier         object
pick5_1            object
pick5_2            object
pick5_3            object
pick5_4            object
pick5_5            object
winning_numbers    object
dtype: object

No problem, I'll create a mapper dictionary and pass it to the `DataFrame.astype()` method to convert all numbers to `int64`.

In [15]:
mapper = {
    "mega_ball": np.int_,
    "multiplier": np.int_,
    "pick5_1": np.int_,
    "pick5_2": np.int_,
    "pick5_3": np.int_,
    "pick5_4": np.int_,
    "pick5_5": np.int_,
}
data_wide = data_wide.astype(mapper)
data_wide.dtypes

variable
mega_ball           int64
multiplier          int64
pick5_1             int64
pick5_2             int64
pick5_3             int64
pick5_4             int64
pick5_5             int64
winning_numbers    object
dtype: object

Let's move the "winning_numbers" column back to it's original position. I can do this by popping it out of its current position in `data_wide` and inserting it into the first column position in the frame by calling [DataFrame.pop()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pop.html) and passing the `Series` returned by the call directly to [DataFrame.insert()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.insert.html).

In [16]:
label = "winning_numbers"
data_wide.insert(0, label, data_wide.pop(label))
data_wide.head(3)

variable,winning_numbers,mega_ball,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5
draw_date,,,,,,,,
2017-10-31,06 28 31 52 53,12,4,6,28,31,52,53
2017-11-03,10 22 42 61 69,3,2,10,22,42,61,69
2017-11-07,01 54 60 68 69,11,4,1,54,60,68,69


I'd like "draw_date" to revert to a column so I'll call [DataFrame.reset_index()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html) and fall back to the default index (`0`, ..., `n - 1`). I'll mutate the existing `DataFrame` rather than creating a new one.

In [17]:
data_wide.reset_index(inplace=True)
data_wide.head(3)

variable,draw_date,winning_numbers,mega_ball,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5
0,2017-10-31,06 28 31 52 53,12,4,6,28,31,52,53
1,2017-11-03,10 22 42 61 69,3,2,10,22,42,61,69
2,2017-11-07,01 54 60 68 69,11,4,1,54,60,68,69


There's no need to retain the axis name "variable" so I'll drop it by calling the [`DataFrame.rename_axis()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename_axis.html) method.

In [18]:
data_wide.rename_axis(None, axis=1, inplace=True)
data_wide.head(3)

,draw_date,winning_numbers,mega_ball,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5
0,2017-10-31,06 28 31 52 53,12,4,6,28,31,52,53
1,2017-11-03,10 22 42 61 69,3,2,10,22,42,61,69
2,2017-11-07,01 54 60 68 69,11,4,1,54,60,68,69


The original Mega Millions CSV files sort the winning numbers by "draw_date" in descending order. I am going to do the same by calling the `DataFrame.sort_values()` method.

In [19]:
data_wide.sort_values(
    "draw_date", ascending=False, ignore_index=True, inplace=True
)
data_wide.head(3)

,draw_date,winning_numbers,mega_ball,multiplier,pick5_1,pick5_2,pick5_3,pick5_4,pick5_5
0,2024-05-17,08 17 40 60 70,3,2,8,17,40,60,70
1,2024-05-14,13 19 43 62 64,6,3,13,19,43,62,64
2,2024-05-10,13 22 26 32 65,18,4,13,22,26,32,65


 Time to review the frame's vitals:

In [20]:
data_wide.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 684 entries, 0 to 683
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   draw_date        684 non-null    datetime64[ns]
 1   winning_numbers  684 non-null    object        
 2   mega_ball        684 non-null    int64         
 3   multiplier       684 non-null    int64         
 4   pick5_1          684 non-null    int64         
 5   pick5_2          684 non-null    int64         
 6   pick5_3          684 non-null    int64         
 7   pick5_4          684 non-null    int64         
 8   pick5_5          684 non-null    int64         
dtypes: datetime64[ns](1), int64(7), object(1)
memory usage: 48.2+ KB


### Write to file

Let's conclude our discussion of `DataFrame` reshaping by writing `data_wide` to a CSV file. We will make use of the data file in a future lesson. 🙂

In [21]:
filepath = "./data/mega_millions_pick5_cols-2017_24.csv"
data_wide.to_csv(filepath, index=False)